# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.config import Settings
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [3]:
# API keys are read from a .env file in this folder (never commit it):
# OPENAI_API_KEY="YOUR_KEY"         - used by the agent in Part 2
# CHROMA_OPENAI_API_KEY="YOUR_KEY"  - used by ChromaDB to create embeddings
# TAVILY_API_KEY="YOUR_KEY"         - used for web search in Part 2

In [4]:
# Load the API keys from .env into environment variables
load_dotenv()

True

### VectorDB Instance

In [5]:
# Persistent client: the database is saved to the "chromadb" folder
# so Part 2 can load it without re-indexing.
# Telemetry is disabled to avoid a harmless posthog version warning.
chroma_client = chromadb.PersistentClient(
    path="chromadb",
    settings=Settings(anonymized_telemetry=False),
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


### Collection

In [6]:
# OpenAI embedding function. Part 2 must use the same model,
# otherwise query embeddings won't match the stored ones.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    model_name="text-embedding-ada-002",
)

In [7]:
# Start from a clean collection so re-running the notebook
# doesn't create duplicates or clash with an older embedding setup.
try:
    chroma_client.delete_collection(name="udaplay")
except Exception:
    pass  # collection didn't exist yet

collection = chroma_client.create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Add documents

In [8]:
# Each JSON file in games/ becomes one document in the collection
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Text that gets embedded: platform, name, year and description
    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"

    # File name without extension (e.g. "001") is the document ID
    doc_id = os.path.splitext(file_name)[0]

    # Full game record is stored as metadata for use in Part 2
    collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game],
    )

Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


## Tests

Check that all 15 games were added.

In [9]:
print(collection.count())  # expected: 15

15


Preview a few stored documents to confirm the indexed text looks right.

In [10]:
# Show the first 3 documents in the collection
sample = collection.peek(limit=3)
for doc_id, doc in zip(sample["ids"], sample["documents"]):
    print(doc_id, "->", doc)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


001 -> [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
002 -> [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.
003 -> [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.


Semantic search test: the correct game should appear first for each question.
The number shown is the distance, where **lower means a closer match**.

In [11]:
# Run sample questions and print the top 3 matches for each
questions = [
    "When was Pokémon Gold and Silver released?",
    "first 3D Mario platformer",
    "racing simulator on PlayStation",
]

for q in questions:
    res = collection.query(query_texts=[q], n_results=3)
    print(f"\nQ: {q}")
    for meta, dist in zip(res["metadatas"][0], res["distances"][0]):
        print(f"  {dist:.3f}  {meta['Name']} ({meta['Platform']}, {meta['YearOfRelease']})")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



Q: When was Pokémon Gold and Silver released?
  0.227  Pokémon Gold and Silver (Game Boy Color, 1999)
  0.295  Pokémon Ruby and Sapphire (Game Boy Advance, 2002)
  0.442  Super Mario 64 (Nintendo 64, 1996)

Q: first 3D Mario platformer
  0.212  Super Mario 64 (Nintendo 64, 1996)
  0.269  Super Mario World (Super Nintendo Entertainment System (SNES), 1990)
  0.373  Mario Kart 8 Deluxe (Nintendo Switch, 2017)

Q: racing simulator on PlayStation
  0.228  Gran Turismo 5 (PlayStation 3, 2010)
  0.230  Gran Turismo (PlayStation 1, 1997)
  0.398  Mario Kart 8 Deluxe (Nintendo Switch, 2017)
